# EatAI: Build-Your-Own Meal Solver (v4)



## Model
**1 Entree + free Grain + free Beans + free Toppings + optional paid Extras (guac/queso)**

In [1]:
import json
import itertools
import time
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field
from copy import deepcopy

In [2]:
json_path = "/Users/abdullahalfuraih/Downloads/restaurants.menu_item_variations.json"
with open(json_path, encoding='utf-8') as f:
    item_list = json.load(f)

byo_items = [item for item in item_list if item['restaurant_name'] == 'Chipotle']
print(f"Loaded {len(byo_items)} Chipotle items")

Loaded 99 Chipotle items


---
## 2. Fix Topping Prices

At Chipotle, most toppings are **free** when part of a bowl/burrito/salad build.
Only guac and queso are upcharged. The DB has them all priced as standalone sides.

We fix this by creating modified copies with price=0 for included toppings.
Original data is untouched.

In [3]:
# Items that cost EXTRA (not included free with entree)
PAID_EXTRAS = ['Guacamole', 'Queso']

# Everything else in "Extras and Toppings" is free when added to a bowl/burrito
# We create modified copies with price=0 for the free ones

byo_items_fixed = []
price_fixes = 0

for item in byo_items:
    if item['category'] == 'Extras and Toppings' and item['menu_item_name'] not in PAID_EXTRAS:
        fixed = deepcopy(item)
        old_price = fixed['price']
        fixed['price'] = 0.0
        fixed['_original_price'] = old_price  # keep for reference
        byo_items_fixed.append(fixed)
        price_fixes += 1
    else:
        byo_items_fixed.append(deepcopy(item))

print(f"Fixed {price_fixes} topping prices to $0.00")
print(f"\nPaid extras (still have price):")
for item in byo_items_fixed:
    if item['category'] == 'Extras and Toppings' and item['price'] > 0:
        print(f"  {item['menu_item_name']}: ${item['price']}")

print(f"\nFree toppings:")
for item in byo_items_fixed:
    if item['category'] == 'Extras and Toppings' and item['price'] == 0:
        print(f"  {item['menu_item_name']} (was ${item.get('_original_price', '?')})")

Fixed 15 topping prices to $0.00

Paid extras (still have price):
  Guacamole: $2.5
  Queso: $2.5

Free toppings:
  Black Beans (was $2.5)
  Brown Rice (was $2.5)
  Corn Salsa (was $2.5)
  Cheese (was $1.25)
  Cilantro-Lime Cauliflower Rice (was $2.5)
  Fajita Veggies (was $2.5)
  Red Tomatillo Salsa (was $1.25)
  Vinaigrette Dressing (was $1.5)
  Lettuce (was $1.25)
  Sour Cream (was $1.25)
  Green Tomatillo Salsa (was $1.5)
  Pinto Beans (was $1.5)
  Tomato Salsa (was $1.5)
  White Rice (was $2.5)
  Supergreens (was $2.5)


---
## 3. Config

### Performance fix: Chips & Drinks removed from core solver

Chips and drinks aren't part of the bowl/burrito *build* — they're separate items.
Including them multiplied the search space by 8×19 = 152x for no real benefit.

If a user wants chips or a drink, that's handled by the ADD_SIDE template, not BUILD_FULL_MEAL.

**Search space: 57 entrees × (4 grain × 3 beans × 79 toppings × 3 paid extras) = ~57 × 948 ≈ 54K**  
vs v3: 57 × 144K = 8.2M

In [4]:
@dataclass
class CategoryRule:
    category_name: str
    min_picks: int
    max_picks: int
    is_required: bool
    source_categories: List[str] = field(default_factory=list)
    source_item_names: List[str] = field(default_factory=list)

@dataclass
class RestaurantBuildConfig:
    restaurant_name: str
    category_rules: List[CategoryRule]
    
    def get_rule(self, role: str) -> Optional[CategoryRule]:
        for rule in self.category_rules:
            if rule.category_name == role:
                return rule
        return None
    
    def required_roles(self) -> List[str]:
        return [r.category_name for r in self.category_rules if r.is_required]
    
    def optional_roles(self) -> List[str]:
        return [r.category_name for r in self.category_rules if not r.is_required]
    
    def get_items_for_role(self, role: str, all_items: List[dict]) -> List[dict]:
        rule = self.get_rule(role)
        if not rule:
            return []
        candidates = [item for item in all_items if item['category'] in rule.source_categories]
        if rule.source_item_names:
            candidates = [item for item in candidates if item['menu_item_name'] in rule.source_item_names]
        return candidates


ENTREE_CATEGORIES = ['Burrito Bowls', 'Burritos', 'Tacos', 'Quesadillas', 'Salads', 'Lifestyle Bowls']

GRAIN_ITEMS = ['White Rice', 'Brown Rice', 'Cilantro-Lime Cauliflower Rice']
BEAN_ITEMS = ['Black Beans', 'Pinto Beans']
TOPPING_ITEMS = [
    'Corn Salsa', 'Cheese', 'Fajita Veggies', 'Red Tomatillo Salsa',
    'Vinaigrette Dressing', 'Lettuce', 'Sour Cream',
    'Green Tomatillo Salsa', 'Tomato Salsa', 'Supergreens'
]
PAID_EXTRA_ITEMS = ['Guacamole', 'Queso']  # These actually cost money

CHIPOTLE_CONFIG = RestaurantBuildConfig(
    restaurant_name="Chipotle",
    category_rules=[
        CategoryRule(
            category_name="Entree",
            min_picks=1, max_picks=1, is_required=True,
            source_categories=ENTREE_CATEGORIES,
        ),
        CategoryRule(
            category_name="Grain",
            min_picks=0, max_picks=1, is_required=False,
            source_categories=['Extras and Toppings'],
            source_item_names=GRAIN_ITEMS,
        ),
        CategoryRule(
            category_name="Beans",
            min_picks=0, max_picks=1, is_required=False,
            source_categories=['Extras and Toppings'],
            source_item_names=BEAN_ITEMS,
        ),
        CategoryRule(
            category_name="Toppings",
            min_picks=0, max_picks=2, is_required=False,
            source_categories=['Extras and Toppings'],
            source_item_names=TOPPING_ITEMS,
        ),
        CategoryRule(
            category_name="Paid Extras",
            min_picks=0, max_picks=1, is_required=False,
            source_categories=['Extras and Toppings'],
            source_item_names=PAID_EXTRA_ITEMS,
        ),
    ]
)

# Verify and show combo counts
print("Items per role:")
total_combos = 1
for rule in CHIPOTLE_CONFIG.category_rules:
    items = CHIPOTLE_CONFIG.get_items_for_role(rule.category_name, byo_items_fixed)
    req = "REQUIRED" if rule.is_required else "optional"
    if not rule.is_required:
        from math import comb
        n_combos = sum(comb(len(items), r) for r in range(rule.min_picks, rule.max_picks + 1))
        total_combos *= n_combos
        print(f"  {rule.category_name:<12} ({req}): {len(items):>3} items  [pick {rule.min_picks}-{rule.max_picks}]  → {n_combos} combos")
    else:
        print(f"  {rule.category_name:<12} ({req}): {len(items):>3} items  [pick {rule.min_picks}-{rule.max_picks}]")

entree_count = len(CHIPOTLE_CONFIG.get_items_for_role('Entree', byo_items_fixed))
print(f"\nTotal search space: {entree_count} × {total_combos} = {entree_count * total_combos:,}")

Items per role:
  Entree       (REQUIRED):  57 items  [pick 1-1]
  Grain        (optional):   3 items  [pick 0-1]  → 4 combos
  Beans        (optional):   2 items  [pick 0-1]  → 3 combos
  Toppings     (optional):  10 items  [pick 0-2]  → 56 combos
  Paid Extras  (optional):   2 items  [pick 0-1]  → 3 combos

Total search space: 57 × 2016 = 114,912


---
## 4. Constraints & Profiles

In [5]:
@dataclass
class UserConstraints:
    max_price: float = 15.0
    max_calories: int = 900
    min_protein: int = 0
    max_sodium: int = 99999
    exclude_allergens: List[str] = field(default_factory=list)

@dataclass
class ProfileWeights:
    w_protein: float = 0.35
    w_calories: float = 0.25
    w_price: float = 0.25
    w_completeness: float = 0.15

PROFILES = {
    "bodybuilding": ProfileWeights(w_protein=0.45, w_calories=0.20, w_price=0.15, w_completeness=0.20),
    "lean":         ProfileWeights(w_protein=0.35, w_calories=0.35, w_price=0.15, w_completeness=0.15),
    "balanced":     ProfileWeights(w_protein=0.25, w_calories=0.25, w_price=0.25, w_completeness=0.25),
    "budget":       ProfileWeights(w_protein=0.10, w_calories=0.25, w_price=0.50, w_completeness=0.15),
}

test_constraints = UserConstraints(
    max_price=14.00,
    max_calories=800,
    min_protein=30,
)

print(f"Test: max ${test_constraints.max_price}, max {test_constraints.max_calories} cal, min {test_constraints.min_protein}g protein")

Test: max $14.0, max 800 cal, min 30g protein


---
## 5. Solver + Scoring

In [6]:
def meal_totals(items: List[dict]) -> dict:
    totals = {'price': 0.0, 'calories': 0, 'protein': 0, 'total_fat': 0, 'total_carbohydrates': 0, 'sodium': 0}
    for item in items:
        totals['price'] += item.get('price', 0)
        ni = item.get('nutrition_info', {})
        for key in ['calories', 'protein', 'total_fat', 'total_carbohydrates', 'sodium']:
            totals[key] += ni.get(key, 0)
    return totals

def passes_constraints(totals: dict, c: UserConstraints) -> bool:
    return (totals['price'] <= c.max_price and
            totals['calories'] <= c.max_calories and
            totals['protein'] >= c.min_protein and
            totals['sodium'] <= c.max_sodium)

def category_combos(items: List[dict], min_p: int, max_p: int) -> List[Tuple]:
    if max_p == -1: max_p = len(items)
    combos = []
    for r in range(min_p, max_p + 1):
        combos.extend(itertools.combinations(items, r))
    return combos

def completeness_score(meal: dict) -> float:
    """Score how much a meal resembles a real Chipotle order."""
    entree_cat = meal.get('entree_category', '')
    
    # Already-complete entrees (rice/beans/toppings baked into nutrition)
    if entree_cat in ['Burritos', 'Tacos', 'Quesadillas', 'Lifestyle Bowls']:
        return 0.80
    
    # Bowls and salads NEED add-ons
    score = 0.0
    addon_names = [i['menu_item_name'] for i in meal['items'][1:]]
    
    if any(n in GRAIN_ITEMS for n in addon_names):   score += 0.30
    if any(n in BEAN_ITEMS for n in addon_names):    score += 0.25
    topping_ct = sum(1 for n in addon_names if n in TOPPING_ITEMS)
    if topping_ct >= 1: score += 0.25
    if topping_ct >= 2: score += 0.20
    
    return score

In [7]:
def solve_byo_meals(
    all_items: List[dict],
    config: RestaurantBuildConfig,
    constraints: UserConstraints,
    verbose: bool = True,
) -> List[dict]:
    required_roles = config.required_roles()
    optional_roles = config.optional_roles()
    
    entree_items = config.get_items_for_role(required_roles[0], all_items)
    
    # Max optional protein for feasibility pruning
    all_opt = []
    for role in optional_roles:
        all_opt.extend(config.get_items_for_role(role, all_items))
    max_opt_protein = sum(i.get('nutrition_info', {}).get('protein', 0) for i in all_opt)
    
    # Generate optional combos
    opt_combo_lists = []
    for role in optional_roles:
        rule = config.get_rule(role)
        items = config.get_items_for_role(role, all_items)
        opt_combo_lists.append(category_combos(items, rule.min_picks, rule.max_picks))
    
    opt_products = list(itertools.product(*opt_combo_lists))
    n_opt = len(opt_products)
    
    if verbose:
        print(f"Entrees: {len(entree_items)} | Optional combos: {n_opt:,} | Search space: {len(entree_items)*n_opt:,}")
    
    valid_meals = []
    pruned = 0
    checked = 0
    
    for entree in entree_items:
        e_price = entree.get('price', 0)
        e_cal = entree.get('nutrition_info', {}).get('calories', 0)
        e_pro = entree.get('nutrition_info', {}).get('protein', 0)
        
        if e_price > constraints.max_price or e_cal > constraints.max_calories:
            pruned += n_opt; continue
        if constraints.min_protein > 0 and e_pro + max_opt_protein < constraints.min_protein:
            pruned += n_opt; continue
        
        for opt_combo in opt_products:
            opt_items = [item for picks in opt_combo for item in picks]
            full_items = [entree] + opt_items
            totals = meal_totals(full_items)
            checked += 1
            
            if passes_constraints(totals, constraints):
                valid_meals.append({
                    'items': full_items,
                    'totals': totals,
                    'item_names': [i['menu_item_name'] for i in full_items],
                    'entree': entree['menu_item_name'],
                    'entree_category': entree['category'],
                })
    
    if verbose:
        total = len(entree_items) * n_opt
        print(f"Checked: {checked:,} | Pruned: {pruned:,} ({pruned/max(total,1)*100:.0f}%) | Valid: {len(valid_meals):,}")
    
    return valid_meals


valid_meals = solve_byo_meals(byo_items_fixed, CHIPOTLE_CONFIG, test_constraints)

Entrees: 57 | Optional combos: 2,016 | Search space: 114,912
Checked: 102,816 | Pruned: 12,096 (11%) | Valid: 43,455


In [8]:
def score_and_rank(meals, weights, top_n=3, diverse=True):
    if not meals:
        print("No valid meals."); return []
    
    proteins = [m['totals']['protein'] for m in meals]
    calories = [m['totals']['calories'] for m in meals]
    prices   = [m['totals']['price'] for m in meals]
    
    def norm(val, lo, hi, higher_better=True):
        if hi == lo: return 0.5
        n = (val - lo) / (hi - lo)
        return n if higher_better else (1.0 - n)
    
    for meal in meals:
        t = meal['totals']
        ps  = norm(t['protein'],  min(proteins), max(proteins), True)
        cs  = norm(t['calories'], min(calories), max(calories), False)
        prs = norm(t['price'],    min(prices),   max(prices),   False)
        comp = completeness_score(meal)
        
        meal['score'] = (weights.w_protein * ps + weights.w_calories * cs +
                         weights.w_price * prs + weights.w_completeness * comp)
        meal['score_breakdown'] = {
            'protein': round(ps, 3), 'calories': round(cs, 3),
            'price': round(prs, 3), 'completeness': round(comp, 3),
        }
    
    meals.sort(key=lambda m: m['score'], reverse=True)
    
    if diverse:
        seen = set()
        result = []
        for meal in meals:
            if meal['entree'] not in seen:
                seen.add(meal['entree'])
                result.append(meal)
            if len(result) >= top_n: break
        return result
    return meals[:top_n]

---
## 6. Results

In [9]:
def display_meal(meal, rank, constraints, profile_name=""):
    t = meal['totals']
    sb = meal.get('score_breakdown', {})
    
    print(f"\n{'─'*70}")
    print(f"  MEAL #{rank}" + (f"  ({profile_name})" if profile_name else "") + f"  [score: {meal.get('score', 0):.3f}]")
    print(f"{'─'*70}")
    
    for item in meal['items']:
        name = item['menu_item_name']
        price = item.get('price', 0)
        cal = item['nutrition_info']['calories']
        pro = item['nutrition_info']['protein']
        price_str = f"${price:<5.2f}" if price > 0 else "FREE "
        print(f"    {name:<35} {price_str} | {cal:>4} cal | {pro:>2}g pro")
    
    print(f"\n    TOTAL: ${t['price']:.2f} | {t['calories']} cal | {t['protein']}g pro | {t['total_fat']}g fat | {t['total_carbohydrates']}g carb")
    print(f"    Scores: pro={sb.get('protein','?')}, cal={sb.get('calories','?')}, price={sb.get('price','?')}, complete={sb.get('completeness','?')}")
    
    if t['price'] > constraints.max_price * 0.9:
        print(f"    ⚠  Near budget limit (${t['price']:.2f} / ${constraints.max_price:.2f})")
    if t['calories'] > constraints.max_calories * 0.9:
        print(f"    ⚠  Near calorie limit ({t['calories']} / {constraints.max_calories})")

In [10]:
for profile_name, weights in PROFILES.items():
    print(f"\n{'='*70}")
    print(f"  PROFILE: {profile_name.upper()}")
    print(f"  Weights: pro={weights.w_protein}, cal={weights.w_calories}, price={weights.w_price}, complete={weights.w_completeness}")
    print(f"  Constraints: max ${test_constraints.max_price}, max {test_constraints.max_calories} cal, min {test_constraints.min_protein}g pro")
    print(f"{'='*70}")
    
    fresh = solve_byo_meals(byo_items_fixed, CHIPOTLE_CONFIG, test_constraints, verbose=False)
    top = score_and_rank(fresh, weights, top_n=3, diverse=True)
    
    if not top:
        print("  No valid meals!")
    else:
        for i, meal in enumerate(top, 1):
            display_meal(meal, i, test_constraints, profile_name)


  PROFILE: BODYBUILDING
  Weights: pro=0.45, cal=0.2, price=0.15, complete=0.2
  Constraints: max $14.0, max 800 cal, min 30g pro

──────────────────────────────────────────────────────────────────────
  MEAL #1  (bodybuilding)  [score: 0.729]
──────────────────────────────────────────────────────────────────────
    Chicken Burrito Bowl                $8.50  |  190 cal | 32g pro
    Cilantro-Lime Cauliflower Rice      FREE  |   40 cal |  3g pro
    Black Beans                         FREE  |  120 cal |  7g pro
    Corn Salsa                          FREE  |   80 cal |  3g pro
    Cheese                              FREE  |  100 cal |  8g pro

    TOTAL: $8.50 | 530 cal | 53g pro | 20g fat | 46g carb
    Scores: pro=0.742, cal=0.443, price=0.714, complete=1.0

──────────────────────────────────────────────────────────────────────
  MEAL #2  (bodybuilding)  [score: 0.719]
──────────────────────────────────────────────────────────────────────
    Chicken Salad                       $9.5

---
## 7. Sanity Check

In [11]:
print("Top meal per profile\n")
print(f"{'Profile':<13} {'Entree':<28} {'Add-ons':<40} {'$':>6} {'Cal':>5} {'Pro':>5}")
print("─" * 100)

for pname, w in PROFILES.items():
    fresh = solve_byo_meals(byo_items_fixed, CHIPOTLE_CONFIG, test_constraints, verbose=False)
    top = score_and_rank(fresh, w, top_n=1, diverse=True)
    if top:
        t = top[0]['totals']
        addons = ', '.join(top[0]['item_names'][1:]) or '(none)'
        print(f"{pname:<13} {top[0]['entree']:<28} {addons:<40} ${t['price']:>5.2f} {t['calories']:>5} {t['protein']:>4}g")

Top meal per profile

Profile       Entree                       Add-ons                                       $   Cal   Pro
────────────────────────────────────────────────────────────────────────────────────────────────────
bodybuilding  Chicken Burrito Bowl         Cilantro-Lime Cauliflower Rice, Black Beans, Corn Salsa, Cheese $ 8.50   530   53g
lean          Chicken Burrito Bowl         Cilantro-Lime Cauliflower Rice, Black Beans, Cheese, Lettuce $ 8.50   460   51g
balanced      Chicken Burrito Bowl         Cilantro-Lime Cauliflower Rice, Black Beans, Cheese, Lettuce $ 8.50   460   51g
budget        Chicken Burrito Bowl         Cilantro-Lime Cauliflower Rice, Black Beans, Lettuce, Green Tomatillo Salsa $ 8.50   375   44g


---
## 8. Performance

In [12]:
runs = 50
start = time.time()
for _ in range(runs):
    meals = solve_byo_meals(byo_items_fixed, CHIPOTLE_CONFIG, test_constraints, verbose=False)
    _ = score_and_rank(meals, PROFILES['balanced'], top_n=3)
elapsed = time.time() - start

print(f"{runs} runs in {elapsed:.2f}s")
print(f"Average per query: {elapsed/runs*1000:.1f}ms")

#test later

50 runs in 15843.54s
Average per query: 316870.7ms


---
## 9. Next Steps

- [ ] Review results — do they look like real Chipotle orders with rice + beans + toppings?
- [ ] Fix the underlying data (set free topping prices to 0 in the DB)
- [ ] Chips & drinks can be added via ADD_SIDE template separately
- [ ] Test with other BYO restaurants
- [ ] Connect to LangChain BUILD_FULL_MEAL template
- [ ] Long term: auto-detect which toppings are free vs paid per restaurant